# 1. 데이터 로드

In [ ]:
! pip install evaluate nltk rouge_score

In [ ]:
! pip install bert_score

In [ ]:
import sys
import os
import time
import torch

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from collections import Counter

import evaluate
from tqdm import tqdm

In [ ]:
df_mimic3 = pd.read_csv("./mimic3/df_mimic3_prompt3_gemma27B.csv")
print(len(df_mimic3))

In [ ]:
df_mimic3['t:LLM_clinical_note'][0]

# 2. 요약 결과 점검

In [ ]:
print("na수와 no clinical note수가 같으면 굳")
print(len(df_mimic3[df_mimic3['t:clinical_note'].isna()]))
print(len(df_mimic3[df_mimic3['t:LLM_clinical_note']=="no clinical note"]))

In [ ]:
df = df_mimic3[~df_mimic3['t:clinical_note'].isna()]
df = df.sample(n=300, random_state=42).reset_index(drop=True)
refs  = df["t:clinical_note"].tolist()
preds = df["t:LLM_clinical_note"].tolist()

# 2) Load the evaluation metric.
rouge      = evaluate.load("rouge")
meteor     = evaluate.load("meteor")
bertscore  = evaluate.load("bertscore")

# 3) Call add_batch for each row and display progress with tqdm.
for pred, ref in tqdm(zip(preds, refs),
                      total=len(df),
                      desc="Evaluating Summaries"):
    # Wrap each value in a list before calling add_batch.
    rouge.add_batch(predictions=[pred], references=[ref])
    meteor.add_batch(predictions=[pred], references=[ref])
    bertscore.add_batch(predictions=[pred], references=[ref])

# 4) Compute the final score.
rouge_res = rouge.compute(use_stemmer=True)
meteor_res = meteor.compute()
bertscore_res = bertscore.compute(lang="en")  # Use 'ko' for Korean text.

# 5) Print the results.
print("▶ ROUGE:", rouge_res)
print("▶ METEOR:", meteor_res)
print("▶ BERTScore:", {
    "precision": bertscore_res["precision"][0],
    "recall":    bertscore_res["recall"][0],
    "f1":        bertscore_res["f1"][0]
})

prompt1 gemma3 27B
▶ ROUGE: {'rouge1': 0.43602609044144347, 'rouge2': 0.21275288164400485, 'rougeL': 0.2901404288951259, 'rougeLsum': 0.4114197497626646}
▶ METEOR: {'meteor': 0.31329734860486697}
▶ BERTScore: {'precision': 0.8497092723846436, 'recall': 0.8096109628677368, 'f1': 0.829175591468811}

prompt3 gemma3 27B
▶ ROUGE: {'rouge1': 0.5948643630193322, 'rouge2': 0.47538977079135997, 'rougeL': 0.4473579709244735, 'rougeLsum': 0.5847480679171801}
▶ METEOR: {'meteor': 0.47149092462533854}
▶ BERTScore: {'precision': 0.8660688400268555, 'recall': 0.8391022682189941, 'f1': 0.8523723483085632}

promp3 llama3 8B
▶ ROUGE: {'rouge1': 0.2955227967391214, 'rouge2': 0.1722889969305778, 'rougeL': 0.22167021076962537, 'rougeLsum': 0.2851029294730303}
▶ METEOR: {'meteor': 0.2545086609461405}
▶ BERTScore: {'precision': 0.8317216634750366, 'recall': 0.7910957932472229, 'f1': 0.8109001517295837}

# 3. train valid test split

In [ ]:
## Determine the train, val, test split (70/15/15), stratified by patient outcome (died_within_48h_of_out_time)
temp = df_mimic3.groupby('traj')['r:reward'].sum()
y = temp.values
X = temp.index.values

Counter(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, stratify=y_test, test_size=0.5)

In [ ]:
# # Drop unneeded meta features
# df_mimic3 = df_mimic3.drop(['t:clinical_note', 'm:charttime', 'm:icustayid'], axis=1)

# Use this branch only when extracting original notes.
df_mimic3 = df_mimic3.drop(['t:LLM_clinical_note', 'm:charttime', 'm:icustayid'], axis=1)
df_mimic3.rename(columns={
    't:clinical_note': 't:LLM_clinical_note'
}, inplace=True)

In [ ]:
train_data = df_mimic3[df_mimic3['traj'].isin(X_train)]
trajectories = train_data['traj'].unique()

val_data = df_mimic3[df_mimic3['traj'].isin(X_val)]
val_trajectories = val_data['traj'].unique()

test_data = df_mimic3[df_mimic3['traj'].isin(X_test)]
test_trajectories = test_data['traj'].unique()

In [ ]:
# Excluded features: max_dose_vaso, SOFA, SIRS, shock_index, cumulated_balance, input_total, input_4hourly, output_total, output_4hourly.
dem_keep_cols = ['o:gender', 'o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:GCS', 'o:HR', 'o:SysBP', 'o:MeanBP', 'o:DiaBP',
       'o:RR', 'o:Temp_C', 'o:FiO2_1', 'o:Potassium', 'o:Sodium', 'o:Chloride',
       'o:Glucose', 'o:Magnesium', 'o:Calcium', 'o:Ionised_Ca', 'o:CO2_mEqL',
       'o:Hb', 'o:WBC_count', 'o:Platelets_count', 'o:PTT', 'o:PT',
       'o:Arterial_pH', 'o:paO2', 'o:paCO2', 'o:HCO3',
       'o:Arterial_lactate', 'o:SOFA', 'o:SIRS', 'o:Shock_Index',
       'o:PaO2_FiO2', 'o:SpO2', 'o:BUN', 'o:Creatinine',
       'o:SGOT', 'o:SGPT', 'o:Total_bili', 'o:Albumin', 'o:INR', 'o:output_4hourly']
text_keep_cols = ['t:LLM_clinical_note']

len(dem_keep_cols)+len(obs_keep_cols) + len(text_keep_cols)

In [ ]:
# Define the features of the full data
text_dim = 1
num_notes = 1
num_dem = 4 #demographics
num_obs = 39
horizon = 21
num_actions = 25
device = 'cpu'

In [ ]:
################################################################
#          FORMAT DATA FOR USE IN SEQUENTIAL MODELS
################################################################

text_keep_cols = ['t:LLM_clinical_note']
dem_keep_cols = ['o:gender', 'o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:GCS', 'o:HR', 'o:SysBP', 'o:MeanBP', 'o:DiaBP',
       'o:RR', 'o:Temp_C', 'o:FiO2_1', 'o:Potassium', 'o:Sodium', 'o:Chloride',
       'o:Glucose', 'o:Magnesium', 'o:Calcium', 'o:Ionised_Ca', 'o:CO2_mEqL',
       'o:Hb', 'o:WBC_count', 'o:Platelets_count', 'o:PTT', 'o:PT',
       'o:Arterial_pH', 'o:paO2', 'o:paCO2', 'o:HCO3',
       'o:Arterial_lactate', 'o:SOFA', 'o:SIRS', 'o:Shock_Index',
       'o:PaO2_FiO2', 'o:SpO2', 'o:BUN', 'o:Creatinine',
       'o:SGOT', 'o:SGPT', 'o:Total_bili', 'o:Albumin', 'o:INR', 'o:output_4hourly']

note_cols = [i for i in train_data.columns if i in text_keep_cols]
dem_cols = [i for i in train_data.columns if i in dem_keep_cols]
obs_cols = [i for i in train_data.columns if i in obs_keep_cols]
ac_cols  = [i for i in train_data.columns if i[:2] == 'a:']
rew_cols = [i for i in train_data.columns if i[:2] == 'r:']
#Assuming discrete actions and scalar rewards:

assert len(obs_cols) > 0, 'No observations present, or observation columns not prefixed with "o:"'
assert len(ac_cols) > 0, 'No actions present, or actions column not prefixed with "a:"'
assert len(rew_cols) > 0, 'No rewards present, or rewards column not prefixed with "r:"'
assert len(ac_cols) == 1, 'Multiple action columns are present when a single action column is expected'
assert len(rew_cols) == 1, 'Multiple reward columns are present when a single reward column is expected'
ac_col = ac_cols[0]
rew_col = rew_cols[0]

## TRAINING DATA
#---------------------------------------------------------------------------
print("Converting Training Data")
print("--"*20)
# train_data[ac_col] = train_data[ac_col]
all_actions = train_data[ac_col].unique()
all_actions.sort()
try:
    all_actions = all_actions.astype(np.int32)
except:
    raise ValueError('Actions are expected to be integers, but are not.')

all_actions

- trainset

In [ ]:
# if not all(all_actions == np.arange(num_actions, dtype=np.int32)):
    # print(Font.red + 'Some actions are missing from data or all action space not properly defined.' + Font.end)
data_trajectory = {}
data_trajectory['text_cols'] = note_cols
data_trajectory['dem_cols'] = dem_cols
data_trajectory['obs_cols'] = obs_cols
data_trajectory['ac_col']  = ac_col
data_trajectory['rew_col'] = rew_col
data_trajectory['num_actions'] = num_actions
data_trajectory['obs_dim'] = len(obs_cols)
data_trajectory['traj'] = {}
data_trajectory['pos_traj'] = []
data_trajectory['neg_traj'] = []

for i in trajectories:
    # bar.update()
    traj_i = train_data[train_data['traj'] == i].sort_values(by='step')
    data_trajectory['traj'][i] = {}
    data_trajectory['traj'][i]['note'] = traj_i[note_cols].values
    data_trajectory['traj'][i]['dem'] = torch.Tensor(traj_i[dem_cols].values).to('cpu')
    data_trajectory['traj'][i]['obs'] = torch.Tensor(traj_i[obs_cols].values).to('cpu')
    data_trajectory['traj'][i]['actions'] = torch.Tensor(traj_i[ac_col].values.astype(np.int32)).to('cpu').long()
    data_trajectory['traj'][i]['rewards'] = torch.Tensor(traj_i[rew_col].values).to('cpu')
    if sum(traj_i[rew_col].values) > 0:
        data_trajectory['pos_traj'].append(i)
    else:
        data_trajectory['neg_traj'].append(i)

In [ ]:
notes = np.full((len(trajectories), horizon, num_notes), "no clinical note", dtype=object)
observations = torch.zeros((len(trajectories), horizon, num_obs))
demographics = torch.zeros((len(trajectories), horizon, num_dem)) 
actions = torch.zeros((len(trajectories), horizon-1, num_actions))
lengths = torch.zeros((len(trajectories)), dtype=torch.int)
times = torch.zeros((len(trajectories), horizon))
rewards = torch.zeros((len(trajectories), horizon))
action_temp = torch.eye(25) #one-hot encoded action (25,25)

for ii, traj in enumerate(trajectories):
    text = data_trajectory['traj'][traj]['note']
    obs = data_trajectory['traj'][traj]['obs']
    dem = data_trajectory['traj'][traj]['dem']
    action = data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    
    lengths[ii] = length
    temp = action_temp[action].squeeze(1)
    
    
    notes[ii] = np.concatenate((text, np.full((horizon-length, 1), "no clinical note")), axis=0)
    observations[ii] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    demographics[ii] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    actions[ii] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    times[ii] = torch.Tensor(range(horizon))
    rewards[ii] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))

In [ ]:
print("이게 있다고?")
notes[lengths==1.0]

In [ ]:
notes = notes[lengths>1.0]
actions = actions[lengths>1.0].to(device)
observations = observations[lengths>1.0].to(device)
demographics = demographics[lengths>1.0].to(device)
times = times[lengths>1.0].to(device)
rewards = rewards[lengths>1.0].to(device)
lengths = lengths[lengths>1.0].to(device)

In [ ]:
print(notes.shape, demographics.shape, observations.shape, actions.shape, lengths.shape, times.shape, rewards.shape)

In [ ]:
# print("Saving off tuples")
# torch.save((notes,demographics,observations,actions,lengths,times,rewards),os.path.join("./mimic3/tuples",'train_set_tuples_with_note'))


torch.save((notes,demographics,observations,actions,lengths,times,rewards),os.path.join("./mimic3/tuples",'train_set_tuples_with_note_no_summerization'))

- validset

In [ ]:
## Validation DATA
#---------------------------------------------------------------------------
print("Converting Validation Data")
print("="*20)
val_data_trajectory = {}
val_data_trajectory['text_cols'] = note_cols
val_data_trajectory['obs_cols'] = obs_cols
val_data_trajectory['dem_cols'] = dem_cols
val_data_trajectory['ac_col']  = ac_col
val_data_trajectory['rew_col'] = rew_col
val_data_trajectory['num_actions'] = num_actions
val_data_trajectory['obs_dim'] = len(obs_cols)
val_data_trajectory['traj'] = {}
val_data_trajectory['pos_traj'] = []
val_data_trajectory['neg_traj'] = []

for j in val_trajectories:
    traj_j = val_data[val_data['traj']==j].sort_values(by='step')
    val_data_trajectory['traj'][j] = {}
    val_data_trajectory['traj'][j]['note'] = traj_j[note_cols].values
    val_data_trajectory['traj'][j]['dem'] = torch.Tensor(traj_j[dem_cols].values).to('cpu')
    val_data_trajectory['traj'][j]['obs'] = torch.Tensor(traj_j[obs_cols].values).to('cpu')
    val_data_trajectory['traj'][j]['actions'] = torch.Tensor(traj_j[ac_col].values.astype(np.int32)).to('cpu').long()
    val_data_trajectory['traj'][j]['rewards'] = torch.Tensor(traj_j[rew_col].values).to('cpu')
    if sum(traj_j[rew_col].values) > 0:
        val_data_trajectory['pos_traj'].append(j)
    else:
        val_data_trajectory['neg_traj'].append(j)

In [ ]:
val_notes = np.full((len(val_trajectories), horizon, num_notes), "no clinical note", dtype=object)
val_obs = torch.zeros((len(val_trajectories), horizon, num_obs))
val_dem = torch.zeros((len(val_trajectories), horizon, num_dem))
val_actions = torch.zeros((len(val_trajectories), horizon-1, num_actions))
val_lengths = torch.zeros((len(val_trajectories)), dtype=torch.int)
val_times = torch.zeros((len(val_trajectories),horizon))
val_rewards = torch.zeros((len(val_trajectories), horizon))
action_temp = torch.eye(25)

for jj, traj in enumerate(val_trajectories):
    text = val_data_trajectory['traj'][traj]['note']
    obs = val_data_trajectory['traj'][traj]['obs']
    dem = val_data_trajectory['traj'][traj]['dem']
    action = val_data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = val_data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    val_lengths[jj] = length
    temp = action_temp[action].squeeze(1)

    val_notes[jj] = np.concatenate((text, np.full((horizon-length, 1), 'no clinical note')), axis=0)
    val_obs[jj] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    val_dem[jj] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    val_actions[jj] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    val_times[jj] = torch.Tensor(range(horizon))
    val_rewards[jj] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))


# Eliminate single transition trajectories...
val_notes = val_notes[val_lengths>1.0]
val_actions = val_actions[val_lengths>1.0].to(device)
val_obs = val_obs[val_lengths>1.0].to(device)
val_dem = val_dem[val_lengths>1.0].to(device)
val_times = val_times[val_lengths>1.0].to(device)
val_rewards = val_rewards[val_lengths>1.0].to(device)
val_lengths = val_lengths[val_lengths>1.0].to(device)

In [ ]:
print(val_notes.shape, val_dem.shape, val_obs.shape, val_actions.shape, val_lengths.shape, val_times.shape, val_rewards.shape)

In [ ]:
# print("Saving off tuples")
# torch.save((val_notes, val_dem,val_obs,val_actions,val_lengths,val_times,val_rewards),os.path.join("./mimic3/tuples","val_set_tuples_with_note"))


torch.save((val_notes, val_dem,val_obs,val_actions,val_lengths,val_times,val_rewards),os.path.join("./mimic3/tuples","val_set_tuples_with_note_no_summerization"))

- testset

In [ ]:
## Test DATA
#---------------------------------------------------------------------------
print("Converting Test Data")
print("+"*20)
test_data_trajectory = {}
test_data_trajectory['text_cols'] = note_cols
test_data_trajectory['obs_cols'] = obs_cols
test_data_trajectory['dem_cols'] = dem_cols
test_data_trajectory['ac_col']  = ac_col
test_data_trajectory['rew_col'] = rew_col
test_data_trajectory['num_actions'] = num_actions
test_data_trajectory['obs_dim'] = len(obs_cols)
test_data_trajectory['traj'] = {}
test_data_trajectory['pos_traj'] = []
test_data_trajectory['neg_traj'] = []

for j in test_trajectories:
    traj_j = test_data[test_data['traj']==j].sort_values(by='step')
    test_data_trajectory['traj'][j] = {}
    test_data_trajectory['traj'][j]['note'] = traj_j[note_cols].values
    test_data_trajectory['traj'][j]['obs'] = torch.Tensor(traj_j[obs_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['dem'] = torch.Tensor(traj_j[dem_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['actions'] = torch.Tensor(traj_j[ac_col].values.astype(np.int32)).to('cpu').long()
    test_data_trajectory['traj'][j]['rewards'] = torch.Tensor(traj_j[rew_col].values).to('cpu')
    if sum(traj_j[rew_col].values) > 0:
        test_data_trajectory['pos_traj'].append(j)
    else:
        test_data_trajectory['neg_traj'].append(j)

test_notes = np.full((len(test_trajectories), horizon, num_notes), "no clinical note", dtype=object)
test_obs = torch.zeros((len(test_trajectories), horizon, num_obs))
test_dem = torch.zeros((len(test_trajectories), horizon, num_dem))
test_actions = torch.zeros((len(test_trajectories), horizon-1, num_actions))
test_lengths = torch.zeros((len(test_trajectories)), dtype=torch.int)
test_times = torch.zeros((len(test_trajectories), horizon))
test_rewards = torch.zeros((len(test_trajectories), horizon))
action_temp = torch.eye(25)

for jj, traj in enumerate(test_trajectories):
    text = test_data_trajectory['traj'][traj]['note']
    obs = test_data_trajectory['traj'][traj]['obs']
    dem = test_data_trajectory['traj'][traj]['dem']
    action = test_data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = test_data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    test_lengths[jj] = length
    temp = action_temp[action].squeeze(1)

    test_notes[jj] = np.concatenate((text, np.full((horizon-length, 1), 'no clinical note')), axis=0)
    test_obs[jj] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    test_dem[jj] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    test_actions[jj] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    test_times[jj] = torch.Tensor(range(horizon))
    test_rewards[jj] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))

In [ ]:
# Eliminate single transition trajectories...
test_notes = test_notes[test_lengths>1.0]
test_actions = test_actions[test_lengths>1.0].to(device)
test_obs = test_obs[test_lengths>1.0].to(device)
test_dem = test_dem[test_lengths>1.0].to(device)
test_times = test_times[test_lengths>1.0].to(device)
#test_mortality = test_mortality[test_lengths>1.0].to(device)
test_rewards = test_rewards[test_lengths>1.0].to(device)
test_lengths = test_lengths[test_lengths>1.0].to(device)

In [ ]:
print(test_notes.shape, test_actions.shape, test_obs.shape, test_dem.shape, test_times.shape, test_rewards.shape, test_lengths.shape)

In [ ]:
# torch.save((test_notes, test_dem, test_obs, test_actions,test_lengths,test_times,test_rewards),os.path.join("./mimic3/tuples","test_set_tuples_with_note"))


torch.save((test_notes, test_dem, test_obs, test_actions,test_lengths,test_times,test_rewards),os.path.join("./mimic3/tuples","test_set_tuples_with_note_no_summerization"))

# 4. convert to RL training form

In [ ]:
from util import ReplayBuffer
import numpy as np

In [ ]:
data_path = {"train":"./mimic3/tuples/train_set_tuples_with_note_no_summerization",
            "val":"./mimic3/tuples/val_set_tuples_with_note_no_summerization",
            "test":"./mimic3/tuples/test_set_tuples_with_note_no_summerization"
             }
buffer_path = "./mimic3/buffer_no_summer/"

state_dim = 39+4 #obs + demog
batch_size = 128
buffer_size = 200000
device = "cuda"

buffer = ReplayBuffer(state_dim=state_dim,
                      batch_size=batch_size,
                      buffer_size=buffer_size,
                      device=device,
                      data_path = data_path,
                      buffer_path = buffer_path
                      )
buffer.load_initial_data(only_test_set=True)

In [ ]:
data_path = {"train":"./mimic3/tuples/train_set_tuples_with_note_no_summerization",
            "val":"./mimic3/tuples/val_set_tuples_with_note_no_summerization",
            "test":"./mimic3/tuples/test_set_tuples_with_note_no_summerization"
             }
buffer_path = "./mimic3/buffer_no_summer/"

state_dim = 39+4 #obs + demog
batch_size = 128
buffer_size = 200000
device = "cuda"

buffer = ReplayBuffer(state_dim=state_dim,
                      batch_size=batch_size,
                      buffer_size=buffer_size,
                      device=device,
                      data_path = data_path,
                      buffer_path = buffer_path
                      )
buffer.load_initial_data(only_test_set=False)

In [ ]:
mimic3_train_buffer = ReplayBuffer(state_dim=state_dim,
                                  batch_size=batch_size,
                                  buffer_size=buffer_size,
                                  device=device,
                                  data_path = data_path,
                                  buffer_path = buffer_path
                                  )
mimic3_train_buffer.load(only_test_set=False)

In [ ]:
mimic3_test_buffer = ReplayBuffer(state_dim=state_dim,
                                  batch_size=batch_size,
                                  buffer_size=buffer_size,
                                  device=device,
                                  data_path = data_path,
                                  buffer_path = buffer_path
                                  )
mimic3_test_buffer.load(only_test_set=True)

# 5. embedding clinical note

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

In [ ]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"  # clinical BERT
BATCH_SIZE = 32
MAX_LENGTH = 512  # Adjust as needed.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3) Define the Dataset and DataLoader.
class ClinicalTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        txt = self.texts[idx] if isinstance(self.texts[idx], str) else ""
        tokenized = self.tokenizer(
            txt,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        # Remove the batch dimension with squeeze.
        return {k: v.squeeze(0) for k, v in tokenized.items()}

# 4) Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=False)
model.to(DEVICE)
model.eval()

In [ ]:
test_note = np.load('./mimic3/buffer/test_note.npy', allow_pickle=True)
test_next_note = np.load('./mimic3/buffer/test_next_note.npy', allow_pickle=True)
train_note = np.load('./mimic3/buffer/train_val_note.npy', allow_pickle=True)
train_next_note = np.load('./mimic3/buffer/train_val_next_note.npy', allow_pickle=True)

test_note = test_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.
test_next_note = test_next_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.
train_note = train_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.
train_next_note = train_next_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.

In [ ]:
# 5) Create the DataLoader.
dataset = ClinicalTextDataset(train_next_note, tokenizer, MAX_LENGTH)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
from tqdm import tqdm
# 6) Extract embeddings.
all_embeddings = []
with torch.no_grad():
    for batch in tqdm(loader):
        # Move the batch tensors to the selected GPU or CPU.
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)
        # pooler_output is the 768-dimensional vector for the [CLS] token.
        # (batch_size, hidden_size)
        batch_emb = outputs.pooler_output  
        all_embeddings.append(batch_emb.cpu())

# 7) Combine the batches into one tensor.
embeddings_tensor = torch.cat(all_embeddings, dim=0)  # (22160, 768)

# 8) Convert to a NumPy array if needed.
embeddings_np = embeddings_tensor.numpy()  # shape (22160, 768)

In [ ]:
embeddings_np.shape

In [ ]:
np.save("./mimic3/buffer/train_val_next_note_embedding.npy", embeddings_np)

# 6. Process External Validation Set

In [ ]:
import pandas as pd
df_mimic3 = pd.read_csv("./mimic3/df_mimic3_prompt3_gemma27B.csv", index_col=0)
df_mimic3_scale_mimic4 = pd.read_csv("./mimic3/sepsis_final_data_withTimes_scaling_by_mimic4.csv")
df_mimic3_scale_snuh = pd.read_csv("./mimic3/sepsis_final_data_withTimes_scaling_by_snuh.csv")

In [ ]:
# Use this code only when extracting original notes.
# df_mimic3 = df_mimic3.drop(['t:LLM_clinical_note'], axis=1)
# df_mimic3.rename(columns={
#     't:clinical_note': 't:LLM_clinical_note'
# }, inplace=True)

In [ ]:
df_mimic3_scale_mimic4["t:LLM_clinical_note"] = df_mimic3["t:LLM_clinical_note"]
df_mimic3_scale_snuh["t:LLM_clinical_note"] = df_mimic3["t:LLM_clinical_note"]

In [ ]:
test_icustayid = df_mimic3_scale_mimic4["m:icustayid"].unique()[:2000]
df_mimic3_scale_mimic4= df_mimic3_scale_mimic4[df_mimic3_scale_mimic4['m:icustayid'].isin(test_icustayid)]

test_icustayid = df_mimic3_scale_snuh["m:icustayid"].unique()[:2000]
df_mimic3_scale_snuh= df_mimic3_scale_snuh[df_mimic3_scale_snuh['m:icustayid'].isin(test_icustayid)]

- mimic4

In [ ]:
from collections import Counter
## Determine the train, val, test split (70/15/15), stratified by patient outcome (died_within_48h_of_out_time)
temp = df_mimic3_scale_mimic4.groupby('traj')['r:reward'].sum()
y = temp.values
X = temp.index.values

Counter(y)

In [ ]:
# Drop unneeded meta features
df_mimic3_scale_mimic4 = df_mimic3_scale_mimic4.drop(['m:clinical_note', 'm:charttime', 'm:icustayid'], axis=1)

In [ ]:
test_data = df_mimic3_scale_mimic4
test_trajectories = df_mimic3_scale_mimic4['traj'].unique()

In [ ]:
# Excluded features: max_dose_vaso, SOFA, SIRS, shock_index, cumulated_balance, input_total, input_4hourly, output_total, output_4hourly.
dem_keep_cols = ['o:gender', 'o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:GCS', 'o:HR', 'o:SysBP', 'o:MeanBP', 'o:DiaBP',
       'o:RR', 'o:Temp_C', 'o:FiO2_1', 'o:Potassium', 'o:Sodium', 'o:Chloride',
       'o:Glucose', 'o:Magnesium', 'o:Calcium', 'o:Ionised_Ca', 'o:CO2_mEqL',
       'o:Hb', 'o:WBC_count', 'o:Platelets_count', 'o:PTT', 'o:PT',
       'o:Arterial_pH', 'o:paO2', 'o:paCO2', 'o:HCO3',
       'o:Arterial_lactate', 'o:SOFA', 'o:SIRS', 'o:Shock_Index',
       'o:PaO2_FiO2', 'o:SpO2', 'o:BUN', 'o:Creatinine',
       'o:SGOT', 'o:SGPT', 'o:Total_bili', 'o:Albumin', 'o:INR', 'o:output_4hourly']
text_keep_cols = ['t:LLM_clinical_note']

len(dem_keep_cols)+len(obs_keep_cols) + len(text_keep_cols)

In [ ]:
# Define the features of the full data
text_dim = 1
num_notes = 1
num_dem = 4 #demographics
num_obs = 39
horizon = 21
num_actions = 25
device = 'cpu'

In [ ]:
import numpy as np
################################################################
#          FORMAT DATA FOR USE IN SEQUENTIAL MODELS
################################################################

text_keep_cols = ['t:LLM_clinical_note']
dem_keep_cols = ['o:gender', 'o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:GCS', 'o:HR', 'o:SysBP', 'o:MeanBP', 'o:DiaBP',
       'o:RR', 'o:Temp_C', 'o:FiO2_1', 'o:Potassium', 'o:Sodium', 'o:Chloride',
       'o:Glucose', 'o:Magnesium', 'o:Calcium', 'o:Ionised_Ca', 'o:CO2_mEqL',
       'o:Hb', 'o:WBC_count', 'o:Platelets_count', 'o:PTT', 'o:PT',
       'o:Arterial_pH', 'o:paO2', 'o:paCO2', 'o:HCO3',
       'o:Arterial_lactate', 'o:SOFA', 'o:SIRS', 'o:Shock_Index',
       'o:PaO2_FiO2', 'o:SpO2', 'o:BUN', 'o:Creatinine',
       'o:SGOT', 'o:SGPT', 'o:Total_bili', 'o:Albumin', 'o:INR', 'o:output_4hourly']

note_cols = [i for i in test_data.columns if i in text_keep_cols]
dem_cols = [i for i in test_data.columns if i in dem_keep_cols]
obs_cols = [i for i in test_data.columns if i in obs_keep_cols]
ac_cols  = [i for i in test_data.columns if i[:2] == 'a:']
rew_cols = [i for i in test_data.columns if i[:2] == 'r:']
#Assuming discrete actions and scalar rewards:

assert len(obs_cols) > 0, 'No observations present, or observation columns not prefixed with "o:"'
assert len(ac_cols) > 0, 'No actions present, or actions column not prefixed with "a:"'
assert len(rew_cols) > 0, 'No rewards present, or rewards column not prefixed with "r:"'
assert len(ac_cols) == 1, 'Multiple action columns are present when a single action column is expected'
assert len(rew_cols) == 1, 'Multiple reward columns are present when a single reward column is expected'
ac_col = ac_cols[0]
rew_col = rew_cols[0]

## TRAINING DATA
#---------------------------------------------------------------------------
print("Converting Training Data")
print("--"*20)
# train_data[ac_col] = train_data[ac_col]
all_actions = test_data[ac_col].unique()
all_actions.sort()
try:
    all_actions = all_actions.astype(np.int32)
except:
    raise ValueError('Actions are expected to be integers, but are not.')

all_actions

In [ ]:
import torch
## Test DATA
#---------------------------------------------------------------------------
print("Converting Test Data")
print("+"*20)
test_data_trajectory = {}
test_data_trajectory['text_cols'] = note_cols
test_data_trajectory['obs_cols'] = obs_cols
test_data_trajectory['dem_cols'] = dem_cols
test_data_trajectory['ac_col']  = ac_col
test_data_trajectory['rew_col'] = rew_col
test_data_trajectory['num_actions'] = num_actions
test_data_trajectory['obs_dim'] = len(obs_cols)
test_data_trajectory['traj'] = {}
test_data_trajectory['pos_traj'] = []
test_data_trajectory['neg_traj'] = []

for j in test_trajectories:
    traj_j = test_data[test_data['traj']==j].sort_values(by='step')
    test_data_trajectory['traj'][j] = {}
    test_data_trajectory['traj'][j]['note'] = traj_j[note_cols].values
    test_data_trajectory['traj'][j]['obs'] = torch.Tensor(traj_j[obs_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['dem'] = torch.Tensor(traj_j[dem_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['actions'] = torch.Tensor(traj_j[ac_col].values.astype(np.int32)).to('cpu').long()
    test_data_trajectory['traj'][j]['rewards'] = torch.Tensor(traj_j[rew_col].values).to('cpu')
    if sum(traj_j[rew_col].values) > 0:
        test_data_trajectory['pos_traj'].append(j)
    else:
        test_data_trajectory['neg_traj'].append(j)

test_notes = np.full((len(test_trajectories), horizon, num_notes), "no clinical note", dtype=object)
test_obs = torch.zeros((len(test_trajectories), horizon, num_obs))
test_dem = torch.zeros((len(test_trajectories), horizon, num_dem))
test_actions = torch.zeros((len(test_trajectories), horizon-1, num_actions))
test_lengths = torch.zeros((len(test_trajectories)), dtype=torch.int)
test_times = torch.zeros((len(test_trajectories), horizon))
test_rewards = torch.zeros((len(test_trajectories), horizon))
action_temp = torch.eye(25)

for jj, traj in enumerate(test_trajectories):
    text = test_data_trajectory['traj'][traj]['note']
    obs = test_data_trajectory['traj'][traj]['obs']
    dem = test_data_trajectory['traj'][traj]['dem']
    action = test_data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = test_data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    test_lengths[jj] = length
    temp = action_temp[action].squeeze(1)

    test_notes[jj] = np.concatenate((text, np.full((horizon-length, 1), 'no clinical note')), axis=0)
    test_obs[jj] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    test_dem[jj] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    test_actions[jj] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    test_times[jj] = torch.Tensor(range(horizon))
    test_rewards[jj] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))

In [ ]:
# Eliminate single transition trajectories...
test_notes = test_notes[test_lengths>1.0]
test_actions = test_actions[test_lengths>1.0].to(device)
test_obs = test_obs[test_lengths>1.0].to(device)
test_dem = test_dem[test_lengths>1.0].to(device)
test_times = test_times[test_lengths>1.0].to(device)
#test_mortality = test_mortality[test_lengths>1.0].to(device)
test_rewards = test_rewards[test_lengths>1.0].to(device)
test_lengths = test_lengths[test_lengths>1.0].to(device)

In [ ]:
print(test_notes.shape, test_actions.shape, test_obs.shape, test_dem.shape, test_times.shape, test_rewards.shape, test_lengths.shape)

In [ ]:
# import os
# torch.save((test_notes, test_dem, test_obs, test_actions,test_lengths,test_times,test_rewards),os.path.join("./mimic3/tuples","test_set_tuples_with_note_scaling_by_mimic4"))


torch.save((test_notes, test_dem, test_obs, test_actions,test_lengths,test_times,test_rewards),os.path.join("./mimic3/tuples","test_set_tuples_with_note_scaling_by_mimic4_no_summerization"))

- snuh

In [ ]:
from collections import Counter
## Determine the train, val, test split (70/15/15), stratified by patient outcome (died_within_48h_of_out_time)
temp = df_mimic3_scale_snuh.groupby('traj')['r:reward'].sum()
y = temp.values
X = temp.index.values

Counter(y)

In [ ]:
# Drop unneeded meta features
df_mimic3_scale_snuh = df_mimic3_scale_snuh.drop(['m:clinical_note', 'm:charttime', 'm:icustayid'], axis=1)

In [ ]:
test_data = df_mimic3_scale_snuh
test_trajectories = df_mimic3_scale_snuh['traj'].unique()

In [ ]:
import numpy as np
################################################################
#          FORMAT DATA FOR USE IN SEQUENTIAL MODELS
################################################################

text_keep_cols = ['t:LLM_clinical_note']
dem_keep_cols = ['o:gender', 'o:re_admission', 'o:age', 'o:Weight_kg']
obs_keep_cols = ['o:GCS', 'o:HR', 'o:SysBP', 'o:MeanBP', 'o:DiaBP',
       'o:RR', 'o:Temp_C', 'o:FiO2_1', 'o:Potassium', 'o:Sodium', 'o:Chloride',
       'o:Glucose', 'o:Magnesium', 'o:Calcium', 'o:Ionised_Ca', 'o:CO2_mEqL',
       'o:Hb', 'o:WBC_count', 'o:Platelets_count', 'o:PTT', 'o:PT',
       'o:Arterial_pH', 'o:paO2', 'o:paCO2', 'o:HCO3',
       'o:Arterial_lactate', 'o:SOFA', 'o:SIRS', 'o:Shock_Index',
       'o:PaO2_FiO2', 'o:SpO2', 'o:BUN', 'o:Creatinine',
       'o:SGOT', 'o:SGPT', 'o:Total_bili', 'o:Albumin', 'o:INR', 'o:output_4hourly']

note_cols = [i for i in test_data.columns if i in text_keep_cols]
dem_cols = [i for i in test_data.columns if i in dem_keep_cols]
obs_cols = [i for i in test_data.columns if i in obs_keep_cols]
ac_cols  = [i for i in test_data.columns if i[:2] == 'a:']
rew_cols = [i for i in test_data.columns if i[:2] == 'r:']
#Assuming discrete actions and scalar rewards:

assert len(obs_cols) > 0, 'No observations present, or observation columns not prefixed with "o:"'
assert len(ac_cols) > 0, 'No actions present, or actions column not prefixed with "a:"'
assert len(rew_cols) > 0, 'No rewards present, or rewards column not prefixed with "r:"'
assert len(ac_cols) == 1, 'Multiple action columns are present when a single action column is expected'
assert len(rew_cols) == 1, 'Multiple reward columns are present when a single reward column is expected'
ac_col = ac_cols[0]
rew_col = rew_cols[0]

## TRAINING DATA
#---------------------------------------------------------------------------
print("Converting Training Data")
print("--"*20)
# train_data[ac_col] = train_data[ac_col]
all_actions = test_data[ac_col].unique()
all_actions.sort()
try:
    all_actions = all_actions.astype(np.int32)
except:
    raise ValueError('Actions are expected to be integers, but are not.')

all_actions

In [ ]:
import torch
## Test DATA
#---------------------------------------------------------------------------
print("Converting Test Data")
print("+"*20)
test_data_trajectory = {}
test_data_trajectory['text_cols'] = note_cols
test_data_trajectory['obs_cols'] = obs_cols
test_data_trajectory['dem_cols'] = dem_cols
test_data_trajectory['ac_col']  = ac_col
test_data_trajectory['rew_col'] = rew_col
test_data_trajectory['num_actions'] = num_actions
test_data_trajectory['obs_dim'] = len(obs_cols)
test_data_trajectory['traj'] = {}
test_data_trajectory['pos_traj'] = []
test_data_trajectory['neg_traj'] = []

for j in test_trajectories:
    traj_j = test_data[test_data['traj']==j].sort_values(by='step')
    test_data_trajectory['traj'][j] = {}
    test_data_trajectory['traj'][j]['note'] = traj_j[note_cols].values
    test_data_trajectory['traj'][j]['obs'] = torch.Tensor(traj_j[obs_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['dem'] = torch.Tensor(traj_j[dem_cols].values).to('cpu')
    test_data_trajectory['traj'][j]['actions'] = torch.Tensor(traj_j[ac_col].values.astype(np.int32)).to('cpu').long()
    test_data_trajectory['traj'][j]['rewards'] = torch.Tensor(traj_j[rew_col].values).to('cpu')
    if sum(traj_j[rew_col].values) > 0:
        test_data_trajectory['pos_traj'].append(j)
    else:
        test_data_trajectory['neg_traj'].append(j)

test_notes = np.full((len(test_trajectories), horizon, num_notes), "no clinical note", dtype=object)
test_obs = torch.zeros((len(test_trajectories), horizon, num_obs))
test_dem = torch.zeros((len(test_trajectories), horizon, num_dem))
test_actions = torch.zeros((len(test_trajectories), horizon-1, num_actions))
test_lengths = torch.zeros((len(test_trajectories)), dtype=torch.int)
test_times = torch.zeros((len(test_trajectories), horizon))
test_rewards = torch.zeros((len(test_trajectories), horizon))
action_temp = torch.eye(25)

for jj, traj in enumerate(test_trajectories):
    text = test_data_trajectory['traj'][traj]['note']
    obs = test_data_trajectory['traj'][traj]['obs']
    dem = test_data_trajectory['traj'][traj]['dem']
    action = test_data_trajectory['traj'][traj]['actions'].view(-1,1)
    reward = test_data_trajectory['traj'][traj]['rewards']
    length = obs.shape[0]
    test_lengths[jj] = length
    temp = action_temp[action].squeeze(1)

    test_notes[jj] = np.concatenate((text, np.full((horizon-length, 1), 'no clinical note')), axis=0)
    test_obs[jj] = torch.cat((obs, torch.zeros((horizon-length, obs.shape[1]), dtype=torch.float)))
    test_dem[jj] = torch.cat((dem, torch.zeros((horizon-length, dem.shape[1]), dtype=torch.float)))
    test_actions[jj] = torch.cat((temp, torch.zeros((horizon-length-1, 25), dtype=torch.float)))
    test_times[jj] = torch.Tensor(range(horizon))
    test_rewards[jj] = torch.cat((reward, torch.zeros((horizon-length), dtype=torch.float)))

In [ ]:
# Eliminate single transition trajectories...
test_notes = test_notes[test_lengths>1.0]
test_actions = test_actions[test_lengths>1.0].to(device)
test_obs = test_obs[test_lengths>1.0].to(device)
test_dem = test_dem[test_lengths>1.0].to(device)
test_times = test_times[test_lengths>1.0].to(device)
#test_mortality = test_mortality[test_lengths>1.0].to(device)
test_rewards = test_rewards[test_lengths>1.0].to(device)
test_lengths = test_lengths[test_lengths>1.0].to(device)

In [ ]:
print(test_notes.shape, test_actions.shape, test_obs.shape, test_dem.shape, test_times.shape, test_rewards.shape, test_lengths.shape)

In [ ]:
# torch.save((test_notes, test_dem, test_obs, test_actions,test_lengths,test_times,test_rewards),os.path.join("./mimic3/tuples","test_set_tuples_with_note_scaling_by_snuh"))


torch.save((test_notes, test_dem, test_obs, test_actions,test_lengths,test_times,test_rewards),os.path.join("./mimic3/tuples","test_set_tuples_with_note_scaling_by_snuh_no_summerization"))

- convert tuples to RL training form

In [ ]:
from util import ReplayBuffer
import numpy as np

In [ ]:
data_path = {"train":"./mimic3/tuples/test_set_tuples_with_note_scaling_by_mimic4_no_summerization",
            "val":"./mimic3/tuples/test_set_tuples_with_note_scaling_by_mimic4_no_summerization",
            "test":"./mimic3/tuples/test_set_tuples_with_note_scaling_by_mimic4_no_summerization"
             }
buffer_path = "./mimic3/buffer_no_summer/scaled_mimic4_"

state_dim = 39+4 #obs + demog
batch_size = 128
buffer_size = 200000
device = "cuda"

buffer = ReplayBuffer(state_dim=state_dim,
                      batch_size=batch_size,
                      buffer_size=buffer_size,
                      device=device,
                      data_path = data_path,
                      buffer_path = buffer_path
                      )
buffer.load_initial_data(only_test_set=True)

In [ ]:
data_path = {"train":"./mimic3/tuples/test_set_tuples_with_note_scaling_by_snuh_no_summerization",
            "val":"./mimic3/tuples/test_set_tuples_with_note_scaling_by_snuh_no_summerization",
            "test":"./mimic3/tuples/test_set_tuples_with_note_scaling_by_snuh_no_summerization"
             }
buffer_path = "./mimic3/buffer_no_summer/scaled_snuh_"

state_dim = 39+4 #obs + demog
batch_size = 128
buffer_size = 200000
device = "cuda"

buffer = ReplayBuffer(state_dim=state_dim,
                      batch_size=batch_size,
                      buffer_size=buffer_size,
                      device=device,
                      data_path = data_path,
                      buffer_path = buffer_path
                      )
buffer.load_initial_data(only_test_set=True)

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

In [ ]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"  # clinical BERT
BATCH_SIZE = 32
MAX_LENGTH = 512  # Adjust as needed.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3) Define the Dataset and DataLoader.
class ClinicalTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        txt = self.texts[idx] if isinstance(self.texts[idx], str) else ""
        tokenized = self.tokenizer(
            txt,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        # Remove the batch dimension with squeeze.
        return {k: v.squeeze(0) for k, v in tokenized.items()}

# 4) Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=False)
model.to(DEVICE)
model.eval()

In [ ]:
test_note_sacled_mimic4 = np.load('./mimic3/buffer/scaled_mimic4_test_note.npy', allow_pickle=True)
test_next_note_scaled_mimic4 = np.load('./mimic3/buffer/scaled_mimic4_test_next_note.npy', allow_pickle=True)
test_note_scaled_snuh = np.load('./mimic3/buffer/scaled_snuh_test_note.npy', allow_pickle=True)
test_next_note_scaled_snuh = np.load('./mimic3/buffer/scaled_snuh_test_next_note.npy', allow_pickle=True)

test_note_sacled_mimic4 = test_note_sacled_mimic4.squeeze(axis=1).tolist()  
test_next_note_scaled_mimic4 = test_next_note_scaled_mimic4.squeeze(axis=1).tolist()  
test_note_scaled_snuh = test_note_scaled_snuh.squeeze(axis=1).tolist()  
test_next_note_scaled_snuh = test_next_note_scaled_snuh.squeeze(axis=1).tolist() 

In [ ]:
# 5) Create the DataLoader.
dataset = ClinicalTextDataset(test_next_note_scaled_snuh, tokenizer, MAX_LENGTH)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
from tqdm import tqdm
# 6) Extract embeddings.
all_embeddings = []
with torch.no_grad():
    for batch in tqdm(loader):
        # Move the batch tensors to the selected GPU or CPU.
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)
        # pooler_output is the 768-dimensional vector for the [CLS] token.
        # (batch_size, hidden_size)
        batch_emb = outputs.pooler_output  
        all_embeddings.append(batch_emb.cpu())

# 7) Combine the batches into one tensor.
embeddings_tensor = torch.cat(all_embeddings, dim=0)  # (22160, 768)

# 8) Convert to a NumPy array if needed.
embeddings_np = embeddings_tensor.numpy()  # shape (22160, 768)

In [ ]:
embeddings_np.shape

In [ ]:
np.save("./mimic3/buffer/scaled_snuh_test_next_note_embedding.npy", embeddings_np)

# note imputation

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

from util import impute_notes, impute_next_notes, stack_notes, stack_next_notes, impute_notes_with_background, impute_next_notes_with_background, bg_stack_note, bg_stack_next_note,extract_background_next_notes, extract_background_states 

In [ ]:
import os

# Specify the GPU index as a zero-based string.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import os
import numpy as np
from tqdm.auto import tqdm
from transformers import pipeline

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

# 0) Configure the environment.
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # HuggingFace Llama 3.1 8B
BATCH_SIZE = 16                          # Adjust the batch size to the available GPU memory.
MAX_LENGTH = 1024                      # Maximum token sequence length.
# SAVE_PATH = "llama3_1-8b_avgpooled_embeddings.npy"

# 1) Create the pipeline.
fe = pipeline(
    "feature-extraction",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=0  # Use device 0 for the GPU or -1 for CPU-only execution.
)

# 2) Define the embedding function.
def embed_with_pipeline_mean_pool(
    texts: list[str],
    batch_size: int,
    max_length: int,
    save_path: str
) -> np.ndarray:
    """Pass texts to the pipeline in batches, mean-pool each sentence's token embeddings, and save and return an array with shape (N, hidden_size)."""
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Feature-extraction"):
        batch = texts[i : i + batch_size]
        # seq_embs is a list of NumPy arrays with shape (max_length, hidden_size).
        seq_embs = fe(
            batch,
            truncation=True,
            padding="max_length",
            max_length=max_length,
            return_tensors="np"
        )
        for seq_emb in seq_embs:
            # Mean-pool over the token dimension, axis 0.
            # print(seq_emb.shape)
            seq_emb = seq_emb.squeeze(0)
            mean_pool = seq_emb.mean(axis=0)  # shape = (hidden_size,)
            all_embs.append(mean_pool)
            # print(mean_pool.shape)
            # print("-----------")

    # (N, hidden_size)
    embs = np.stack(all_embs, axis=0)
    # Save to a .npy file.
    np.save(save_path, embs)
    print(f"Saved avg-pooled embeddings to {save_path}, shape = {embs.shape}")
    return embs

In [ ]:
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"  # Change the model.
BATCH_SIZE = 32
MAX_LENGTH = 512  # Adjust as needed.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3) Define the Dataset and DataLoader.
class ClinicalTextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        txt = self.texts[idx] if isinstance(self.texts[idx], str) else ""
        tokenized = self.tokenizer(
            txt,
            add_special_tokens=True,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        # Remove the batch dimension with squeeze.
        return {k: v.squeeze(0) for k, v in tokenized.items()}

# 4) Load the tokenizer and model.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=False)
model.to(DEVICE)
model.eval()

In [ ]:
def embed_next_notes_bert(
    data,
    tokenizer,
    model,
    device,
    batch_size: int,
    max_length: int,
    save_path: str
) -> np.ndarray:
    """Compute embeddings for train_next_note_stacked, save them to a .npy file, and return the NumPy array.
    
    Args:
        train_next_note_stacked: List of stacked next-note strings.
        tokenizer: HuggingFace tokenizer.
        model: HuggingFace model with pooler_output.
        device: 'cuda' or 'cpu'.
        batch_size: DataLoader batch size.
        max_length: Maximum token length, such as 512 or 1024.
        save_path: Destination .npy file.
    
    Returns:
        embeddings_np: Array with shape (N, hidden_size)."""
    # 1) Create the DataLoader.
    dataset = ClinicalTextDataset(
        data,
        tokenizer,
        max_length
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    all_embeddings = []
    model.to(device)
    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc="Embedding next_notes"):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            # pooler_output contains the [CLS] vectors with shape (batch_size, hidden_size).
            batch_emb = outputs.pooler_output
            all_embeddings.append(batch_emb.cpu())

    # 2) Combine the batches into one tensor.
    embeddings_tensor = torch.cat(all_embeddings, dim=0)  # (N, hidden_size)

    # 3) Convert to a NumPy array.
    embeddings_np = embeddings_tensor.numpy()  # shape (N, hidden_size)

    # 4) Save to a file.
    np.save(save_path, embeddings_np)

In [ ]:
test_note = np.load('./mimic3/buffer_clinical_bert/test_note.npy', allow_pickle=True)
test_next_note = np.load('./mimic3/buffer_clinical_bert/test_next_note.npy', allow_pickle=True)
train_note = np.load('./mimic3/buffer_clinical_bert/train_val_note.npy', allow_pickle=True)
train_next_note = np.load('./mimic3/buffer_clinical_bert/train_val_next_note.npy', allow_pickle=True)

test_done = np.load("./mimic3/buffer_clinical_bert/test_done.npy")
train_done = np.load("./mimic3/buffer_clinical_bert/train_val_done.npy")


test_note = test_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.
test_next_note = test_next_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.
train_note = train_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.
train_next_note = train_next_note.squeeze(axis=1).tolist()  # texts is now a list of 22160 strings.

test_done = test_done.squeeze(axis=1).tolist()
train_done = train_done.squeeze(axis=1).tolist()

In [ ]:
print(len(train_next_note), len(train_done))

- impute

In [ ]:
test_note_imputed      = impute_notes(test_note, test_done)
test_next_note_imputed = impute_next_notes(test_next_note, test_note, test_done, "no clinical note")

train_note_imputed = impute_notes(train_note, train_done)
train_next_note_imputed = impute_next_notes(train_next_note, train_note, train_done, "no clinical note")

- stack

In [ ]:
test_note_stacked      = stack_notes(test_note, test_done, window=7)
test_next_note_stacked = stack_next_notes(test_next_note, test_note, test_done, window=7)

train_note_stacked = stack_notes(train_note, train_done, window=7)
train_next_note_stacked = stack_next_notes(train_next_note, train_note, train_done, window=7)

- initial state background impute

In [ ]:
test_note_bg      = impute_notes_with_background(test_note, test_done)
test_next_note_bg = impute_next_notes_with_background(test_next_note, test_note, test_done)

train_note_bg = impute_notes_with_background(train_note, train_done)
train_next_note_bg = impute_next_notes_with_background(train_next_note, train_note, train_done)

- background + 2 window stack

In [ ]:
test_note_bg_stack      = bg_stack_note(test_note, test_done)
test_next_note_bg_stack = bg_stack_next_note(test_next_note, test_note, test_done)

train_note_bg_stack = bg_stack_note(train_note, train_done)
train_next_note_bg_stack = bg_stack_next_note(train_next_note, train_note, train_done)

- only background

In [ ]:
test_note_only_bg      = extract_background_states(test_note, test_done)
test_next_note_only_bg = extract_background_next_notes(test_next_note, test_note, test_done)

train_note_bg_only_bg = extract_background_states(train_note, train_done)
train_next_note_only_bg = extract_background_next_notes(train_next_note, train_note, train_done)

- raw

test_note


test_next_note 

train_note


train_next_note

In [ ]:
print(len(test_note_stacked), len(test_note))
print(len(test_next_note_stacked), len(test_next_note))
print(len(train_note_stacked), len(train_note))
print(len(train_next_note_stacked), len(train_next_note))

- llama

In [ ]:
dummy_dict = {"./mimic3/buffer_llama/test_note_embedding.npy": test_note,
              "./mimic3/buffer_llama/test_next_note_embedding.npy": test_next_note,
              "./mimic3/buffer_llama/train_val_note_embedding.npy": train_note,
              "./mimic3/buffer_llama/train_val_next_note_embedding.npy": train_next_note}
for path in dummy_dict:
    embed_with_pipeline_mean_pool(
        texts=dummy_dict[path],
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )

In [ ]:
dummy_dict = {"./mimic3/buffer_llama/test_impute_bg_only_note_embedding.npy": test_note_only_bg,
              "./mimic3/buffer_llama/test_impute_bg_only_next_note_embedding.npy": test_next_note_only_bg,
              "./mimic3/buffer_llama/train_val_impute_bg_only_note_embedding.npy": train_note_bg_only_bg,
              "./mimic3/buffer_llama/train_val_impute_bg_only_next_note_embedding.npy": train_next_note_only_bg}
for path in dummy_dict:
    embed_with_pipeline_mean_pool(
        texts=dummy_dict[path],
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )

- bert

In [ ]:
dummy_dict = {"./mimic3/buffer_no_summer_clinical_bert/test_note_embedding.npy": test_note,
              "./mimic3/buffer_no_summer_clinical_bert/test_next_note_embedding.npy": test_next_note,
              "./mimic3/buffer_no_summer_clinical_bert/train_val_note_embedding.npy": train_note,
              "./mimic3/buffer_no_summer_clinical_bert/train_val_next_note_embedding.npy": train_next_note}

for path in dummy_dict:
    embed_next_notes_bert(
        data=dummy_dict[path],
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )

In [ ]:
dummy_dict = {"./mimic3/buffer_clinical_bert/test_stack_w7_note_embedding.npy": test_note_stacked,
              "./mimic3/buffer_clinical_bert/test_stack_w7_next_note_embedding.npy": test_next_note_stacked,
              "./mimic3/buffer_clinical_bert/train_val_stack_w7_note_embedding.npy": train_note_stacked,
              "./mimic3/buffer_clinical_bert/train_val_stack_w7_next_note_embedding.npy": train_next_note_stacked}


for path in dummy_dict:
    embed_next_notes_bert(
        data=dummy_dict[path],
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )

- scaled dataset

In [ ]:
test_note_sacled_mimic4 = np.load('./mimic3/buffer_clinical_bert/scaled_mimic4_test_note.npy', allow_pickle=True)
test_next_note_scaled_mimic4 = np.load('./mimic3/buffer_clinical_bert/scaled_mimic4_test_next_note.npy', allow_pickle=True)
test_note_scaled_snuh = np.load('./mimic3/buffer_clinical_bert/scaled_snuh_test_note.npy', allow_pickle=True)
test_next_note_scaled_snuh = np.load('./mimic3/buffer_clinical_bert/scaled_snuh_test_next_note.npy', allow_pickle=True)

test_done = np.load("./mimic3/buffer_clinical_bert/scaled_mimic4_test_done.npy")

test_note_sacled_mimic4 = test_note_sacled_mimic4.squeeze(axis=1).tolist()  
test_next_note_scaled_mimic4 = test_next_note_scaled_mimic4.squeeze(axis=1).tolist()  
test_note_scaled_snuh = test_note_scaled_snuh.squeeze(axis=1).tolist()  
test_next_note_scaled_snuh = test_next_note_scaled_snuh.squeeze(axis=1).tolist() 

In [ ]:
print(len(test_next_note_scaled_snuh), len(test_done))

- impute

In [ ]:
test_note_sacled_mimic4_imputed      = impute_notes(test_note_sacled_mimic4, test_done)
test_next_note_scaled_mimic4_imputed = impute_next_notes(test_next_note_scaled_mimic4, test_note_sacled_mimic4, test_done, "no clinical note")

test_note_scaled_snuh_imputed = impute_notes(test_note_scaled_snuh, test_done)
test_next_note_scaled_snuh_imputed = impute_next_notes(test_next_note_scaled_snuh, test_note_scaled_snuh, test_done, "no clinical note")

- stack

In [ ]:
test_note_sacled_mimic4_stacked      = stack_notes(test_note_sacled_mimic4, test_done, window=7)
test_next_note_scaled_mimic4_stacked = stack_next_notes(test_next_note_scaled_mimic4, test_note_sacled_mimic4, test_done, window=7)

test_note_scaled_snuh_stacked = stack_notes(test_note_scaled_snuh, test_done, window=7)
test_next_note_scaled_snuh_stacked = stack_next_notes(test_next_note_scaled_snuh, test_note_scaled_snuh, test_done, window=7)

- bg

In [ ]:
test_note_sacled_mimic4_bg      = impute_notes_with_background(test_note_sacled_mimic4, test_done)
test_next_note_scaled_mimic4_bg = impute_next_notes_with_background(test_next_note_scaled_mimic4, test_note_sacled_mimic4, test_done)

test_note_scaled_snuh_bg = impute_notes_with_background(test_note_scaled_snuh, test_done)
test_next_note_scaled_snuh_bg = impute_next_notes_with_background(test_next_note_scaled_snuh, test_note_scaled_snuh, test_done)

- background + stack

In [ ]:
test_note_sacled_mimic4_bg_stack      = bg_stack_note(test_note_sacled_mimic4, test_done)
test_next_note_scaled_mimic4_bg_stack = bg_stack_next_note(test_next_note_scaled_mimic4, test_note_sacled_mimic4, test_done)

test_note_scaled_snuh_bg_stack = bg_stack_note(test_note_scaled_snuh, test_done)
test_next_note_scaled_snuh_bg_stack = bg_stack_next_note(test_next_note_scaled_snuh, test_note_scaled_snuh, test_done)

- only background

In [ ]:
test_note_sacled_mimic4_bg_only      = extract_background_states(test_note_sacled_mimic4, test_done)
test_next_note_scaled_mimic4_bg_only = extract_background_next_notes(test_next_note_scaled_mimic4, test_note_sacled_mimic4, test_done)

test_note_scaled_snuh_bg_only = extract_background_states(test_note_scaled_snuh, test_done)
test_next_note_scaled_snuh_bg_only = extract_background_next_notes(test_next_note_scaled_snuh, test_note_scaled_snuh, test_done)

In [ ]:
print(len(test_note_sacled_mimic4), len(test_note_sacled_mimic4_stacked))
print(len(test_next_note_scaled_mimic4), len(test_next_note_scaled_mimic4_stacked))
print(len(test_note_scaled_snuh), len(test_note_scaled_snuh_stacked))
print(len(test_next_note_scaled_snuh), len(test_next_note_scaled_snuh_stacked))

- llama

In [ ]:
dummy_dict = {"./mimic3/buffer_llama/scaled_mimic4_test_impute_bg_only_note_embedding.npy": test_note_sacled_mimic4_bg_only,
              "./mimic3/buffer_llama/scaled_mimic4_test_impute_bg_only_next_note_embedding.npy": test_next_note_scaled_mimic4_bg_only,
              "./mimic3/buffer_llama/scaled_snuh_test_impute_bg_only_note_embedding.npy": test_note_scaled_snuh_bg_only,
              "./mimic3/buffer_llama/scaled_snuh_test_impute_bg_only_next_note_embedding.npy": test_next_note_scaled_snuh_bg_only}

for path in dummy_dict:
    embed_with_pipeline_mean_pool(
        texts=dummy_dict[path],
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )

In [ ]:
dummy_dict = {"./mimic3/buffer_llama/scaled_mimic4_test_note_embedding.npy": test_note_sacled_mimic4,
              "./mimic3/buffer_llama/scaled_mimic4_test_next_note_embedding.npy": test_next_note_scaled_mimic4,
              "./mimic3/buffer_llama/scaled_snuh_test_note_embedding.npy": test_note_scaled_snuh,
              "./mimic3/buffer_llama/scaled_snuh_test_next_note_embedding.npy": test_next_note_scaled_snuh}

for path in dummy_dict:
    embed_with_pipeline_mean_pool(
        texts=dummy_dict[path],
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )

- bert

In [ ]:
dummy_dict = {"./mimic3/buffer_no_summer_clinical_bert/scaled_mimic4_test_impute_bg_only_note_embedding.npy": test_note_sacled_mimic4_bg_only,
              "./mimic3/buffer_no_summer_clinical_bert/scaled_mimic4_test_impute_bg_only_next_note_embedding.npy": test_next_note_scaled_mimic4_bg_only,
              "./mimic3/buffer_no_summer_clinical_bert/scaled_snuh_test_impute_bg_only_note_embedding.npy": test_note_scaled_snuh_bg_only,
              "./mimic3/buffer_no_summer_clinical_bert/scaled_snuh_test_impute_bg_only_next_note_embedding.npy": test_next_note_scaled_snuh_bg_only}

for path in dummy_dict:
    embed_next_notes_bert(
        data=dummy_dict[path],
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )

In [ ]:
dummy_dict = {"./mimic3/buffer_clinical_bert/scaled_mimic4_test_stack_w7_note_embedding.npy": test_note_sacled_mimic4_stacked,
              "./mimic3/buffer_clinical_bert/scaled_mimic4_test_stack_w7_next_note_embedding.npy": test_next_note_scaled_mimic4_stacked,
              "./mimic3/buffer_clinical_bert/scaled_snuh_test_stack_w7_note_embedding.npy": test_note_scaled_snuh_stacked,
              "./mimic3/buffer_clinical_bert/scaled_snuh_test_stack_w7_next_note_embedding.npy": test_next_note_scaled_snuh_stacked}

for path in dummy_dict:
    embed_next_notes_bert(
        data=dummy_dict[path],
        tokenizer=tokenizer,
        model=model,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        save_path=path
    )